In [1]:
!pip install -U transformers accelerate bitsandbytes sentencepiece pandas tqdm

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed
)

[HAMI-core Msg(790:140352916245824:libvgpu.c:839)]: Initializing.....
[HAMI-core Warn(790:140352916245824:multiprocess_memory_limit.c:548)]: Kick dead proc 248


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100 80GB PCIe


[HAMI-core Msg(790:140352916245824:libvgpu.c:855)]: Initialized


In [4]:
import transformers
import bitsandbytes

print(transformers.__version__)
print(bitsandbytes.__version__)

5.14.1
0.49.2


In [5]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16
)

model.eval()

print("Model loaded successfully.")
print("Model device:", model.device)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully.
Model device: cuda:0


In [7]:
from pathlib import Path

BASE_DIR = Path.cwd().parent

TRAIN_PATH = BASE_DIR / "RUHSOLD_train.tsv"
VAL_PATH = BASE_DIR / "RUHSOLD_validation.tsv"
TEST_PATH = BASE_DIR / "RUHSOLD_test.tsv"

print(TRAIN_PATH)

/home/jovyan/project work/data_analyssis/RUHSOLD_train.tsv


In [8]:
train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t"
)
print(train_df.shape)
print(train_df.head())
print(train_df.columns.tolist())

(6407, 2)
  kia howa hai aap ko allah bless and protect you  aameen  1
0                                         randdi hai       3
1                              smjh to agai thi mjhy       1
2  haan yrr tuny sahi thukayi ki abhi tak lund da...       3
3              rundi ka bacha bharwaaa ptm ka kuttaa       0
4  kbi b nai ay ga vo bcz phr phr yahodi naraz ho...       1
['kia howa hai aap ko allah bless and protect you  aameen', '1']


In [9]:
print(train_df.columns)
print(train_df.head())

Index(['kia howa hai aap ko allah bless and protect you  aameen', '1'], dtype='str')
  kia howa hai aap ko allah bless and protect you  aameen  1
0                                         randdi hai       3
1                              smjh to agai thi mjhy       1
2  haan yrr tuny sahi thukayi ki abhi tak lund da...       3
3              rundi ka bacha bharwaaa ptm ka kuttaa       0
4  kbi b nai ay ga vo bcz phr phr yahodi naraz ho...       1


In [10]:
train_df = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    header=None,
    names=["text", "label"]
)

print(train_df.shape)
print(train_df.columns)
print(train_df.head())

(6408, 2)
Index(['text', 'label'], dtype='str')
                                                text  label
0  kia howa hai aap ko allah bless and protect yo...      1
1                                         randdi hai      3
2                              smjh to agai thi mjhy      1
3  haan yrr tuny sahi thukayi ki abhi tak lund da...      3
4              rundi ka bacha bharwaaa ptm ka kuttaa      0


In [12]:
train_df["label"] = pd.to_numeric(
    train_df["label"],
    errors="raise"
).astype(int)

In [13]:
print(train_df["label"].value_counts().sort_index())

label
0    1537
1    3423
2     500
3     537
4     411
Name: count, dtype: int64


In [14]:
LABEL_NAMES = {
    0: "Abusive/Offensive",
    1: "Normal",
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane"
}

In [15]:
TARGET_CLASSES = {
    2: "Religious Hate",
    3: "Sexism",
    4: "Profane"
}

In [19]:
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

import random

CLASS_WORD_LIMITS = {
    2: (8, 31),   # Religious Hate
    3: (8, 23),   # Sexism
    4: (8, 19)    # Profane
}

def sample_demonstrations(
    dataframe,
    class_id,
    number_of_examples=5,
    random_state=None
):
    min_words, max_words = CLASS_WORD_LIMITS[class_id]

    class_data = dataframe[
        dataframe["label"] == class_id
    ].copy()

    class_data["word_count"] = (
        class_data["text"]
        .astype(str)
        .str.split()
        .str.len()
    )

    eligible_data = class_data[
        class_data["word_count"].between(
            min_words,
            max_words
        )
    ].drop_duplicates(
        subset=["text"]
    )

    if len(eligible_data) < number_of_examples:
        raise ValueError(
            f"Only {len(eligible_data)} eligible examples "
            f"found for class {class_id}."
        )

    demonstrations = eligible_data.sample(
        n=number_of_examples,
        random_state=random_state
    )["text"].tolist()

    random.Random(random_state).shuffle(demonstrations)

    return demonstrations

In [20]:
sexism_examples = sample_demonstrations(
    train_df,
    class_id=3,
    number_of_examples=5,
    random_state=42
)

for i, example in enumerate(sexism_examples, start=1):
    print(f"{i}. {example}")

1. begairat sasti kanjri gulbukhari k sath ek sasti randi 😃 😃 😃 😃
2. bagrat ourat hain allah karein aska hal bhi qandeel baloch ki tara ho aur inshallah aska yahi hal hoga 👍👍👍
3. kya haal hain maya jaalon ki rundi....? 🖕🖕🖕🖕🖕🖕
4. ooooh bc deepika randi ki ma chodunga thode din ruko main btata hoon inko
5. khan sib waqae ahmaq hi kyo ki ap jesi makar awrt si shade jo kye the😢😢


In [21]:
sexism_examples_2 = sample_demonstrations(
    train_df,
    class_id=3,
    number_of_examples=5,
    random_state=43
)

for i, example in enumerate(sexism_examples_2, start=1):
    print(f"{i}. {example}")

1. tum khusray aur dhoghlay insan  tum kush be nahin karo sirf thook kay chaat saktay ho
2. begairat sasti kanjri gulbukhari k sath ek sasti randi 😃 😃 😃 😃
3. kaun iska movie dekhne jata h be iss randi ki ek movie bhi koi na dekhna
4.  shazaib apni baji ko b bejna han aurat march ma agr itne he zruri hai to
5. apni maan k name pe be chona laga gya bilawal hijra


In [45]:
CLASS_DESCRIPTIONS = {
    2: {
        "label": "Religious hate speech",
        "definition": (
            "A post that insults, humiliates, stereotypes, or expresses hostility "
            "towards a person or group because of religion, sect, or religious identity."
        )
    },
    3: {
        "label": "Sexist abusive language",
        "definition": (
            "A post that insults, humiliates, stereotypes, or degrades a person "
            "because of gender."
        )
    },
    4: {
        "label": "Profane language",
        "definition": (
            "A post containing vulgar, obscene, or strongly offensive language, "
            "without necessarily targeting religion or gender."
        )
    }
}

In [46]:
def build_generation_prompt(class_id, demonstrations):
    """
    Build a few-shot prompt for generating one synthetic Roman Urdu post.

    Parameters
    ----------
    class_id : int
        RUHSOLD target label: 2, 3, or 4.

    demonstrations : list[str]
        Real training examples belonging to the target class.

    Returns
    -------
    list[dict]
        Chat-formatted message accepted by the Mistral tokenizer.
    """

    if class_id not in CLASS_DESCRIPTIONS:
        raise ValueError(
            f"Unsupported class_id: {class_id}. "
            f"Expected one of {list(CLASS_DESCRIPTIONS.keys())}."
        )

    if not demonstrations:
        raise ValueError("At least one demonstration is required.")

    class_label = CLASS_DESCRIPTIONS[class_id]["label"]
    class_definition = CLASS_DESCRIPTIONS[class_id]["definition"]

    formatted_examples = "\n\n".join(
        f"Example {index}:\n{text.strip()}"
        for index, text in enumerate(demonstrations, start=1)
    )

    user_prompt = f"""
You are an expert data synthesis assistant for hate speech classification datasets.

You are generating synthetic Roman Urdu social media posts for an academic research dataset.

Dataset label:
{class_label}.

Definition:
{class_definition}

Roman Urdu refers to Urdu written using the English Latin alphabet.

Below are real examples from the dataset.

{formatted_examples}

Generate ONE new Roman Urdu social media post that belongs to the same dataset label.

The generated post should preserve the informal writing style, vocabulary, and linguistic characteristics of the examples while expressing a new idea.

The generated post must be different from all provided examples and must not copy or closely paraphrase any example.

Write primarily in Roman Urdu using the English Latin alphabet. Natural English code-mixing is acceptable when it occurs naturally.

Return only the generated post.
""".strip()

    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    return messages

In [47]:
import re

def uses_allowed_script(text):
    """
    Allows Latin letters, numbers, whitespace, punctuation, and emojis.
    Rejects Urdu, Arabic, and Devanagari characters.
    """
    forbidden_scripts = re.compile(
        r"[\u0600-\u06FF"   # Arabic and Urdu
        r"\u0750-\u077F"
        r"\u08A0-\u08FF"
        r"\u0900-\u097F]"   # Devanagari
    )

    return not bool(forbidden_scripts.search(str(text)))

In [48]:
print(uses_allowed_script(sample))

True


In [49]:
messages = build_generation_prompt(
    class_id=3,
    demonstrations=demonstrations
)

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(formatted_prompt)

<s> [INST] You are an expert data synthesis assistant for hate speech classification datasets.

You are generating synthetic Roman Urdu social media posts for an academic research dataset.

Dataset label:
Sexist abusive language.

Definition:
A post that insults, humiliates, stereotypes, or degrades a person because of gender.

Roman Urdu refers to Urdu written using the English Latin alphabet.

Below are real examples from the dataset.

Example 1:
begairat sasti kanjri gulbukhari k sath ek sasti randi 😃 😃 😃 😃

Example 2:
bagrat ourat hain allah karein aska hal bhi qandeel baloch ki tara ho aur inshallah aska yahi hal hoga 👍👍👍

Example 3:
kya haal hain maya jaalon ki rundi....? 🖕🖕🖕🖕🖕🖕

Example 4:
ooooh bc deepika randi ki ma chodunga thode din ruko main btata hoon inko

Example 5:
khan sib waqae ahmaq hi kyo ki ap jesi makar awrt si shade jo kye the😢😢

Generate ONE new Roman Urdu social media post that belongs to the same dataset label.

The generated post should preserve the informal 

In [50]:
def generate_one_sample(
    messages,
    seed=42,
    max_new_tokens=60,
    temperature=1.0,
    top_p=0.95,
    typical_p=0.8,
    repetition_penalty=1.2
):
    set_seed(seed)

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    input_length = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            typical_p=typical_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = generated_ids[0][input_length:]

    generated_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return generated_text

In [51]:
demonstrations = sample_demonstrations(
    train_df,
    class_id=3,
    number_of_examples=5,
    random_state=42
)

messages = build_generation_prompt(
    class_id=3,
    demonstrations=demonstrations
)

sample = generate_one_sample(
    messages=messages,
    seed=SEED,
    max_new_tokens=60,
    temperature=0.7,
    top_p=0.9,
    typical_p=0.8,
    repetition_penalty=1.2
)

print("\nGenerated sample:")
print(sample)


Generated sample:
meri behan bharatiya janani mandir wali insaan banne ke liye na, tu biwi ka gaddha bana rhe hai 🤮🤮
(My sister, you're supposed to become a Bhar


In [54]:
import re

def extract_generated_post(raw_output):
    """
    Extracts the candidate tweet while preserving the raw output separately.
    """

    text = str(raw_output).strip()

    # Keep only the content before a new-line translation or explanation.
    text = text.split("\n")[0].strip()

    # Remove an English translation beginning in parentheses.
    text = re.split(r"\s*\(", text, maxsplit=1)[0].strip()

    # Remove common unwanted prefixes.
    prefixes = [
        "Generated post:",
        "Generated tweet:",
        "New post:",
        "New tweet:",
        "Output:"
    ]

    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    return re.sub(r"\s+", " ", text)

In [55]:
candidate_post = extract_generated_post(sample)

print("Raw output:")
print(sample)

print("\nExtracted candidate:")
print(candidate_post)

print("\nAllowed script:", uses_allowed_script(candidate_post))
print("Word count:", len(candidate_post.split()))

Raw output:
meri behan bharatiya janani mandir wali insaan banne ke liye na, tu biwi ka gaddha bana rhe hai 🤮🤮
(My sister, you're supposed to become a Bhar

Extracted candidate:
meri behan bharatiya janani mandir wali insaan banne ke liye na, tu biwi ka gaddha bana rhe hai 🤮🤮

Allowed script: True
Word count: 19


In [56]:
pilot_results = []

for seed in [42, 43, 44, 45, 46]:

    demonstrations = sample_demonstrations(
        dataframe=train_df,
        class_id=3,
        number_of_examples=5,
        random_state=seed
    )

    messages = build_generation_prompt(
        class_id=3,
        demonstrations=demonstrations
    )

    raw_output = generate_one_sample(
        messages=messages,
        seed=seed,
        max_new_tokens=60,
        temperature=0.7,
        top_p=0.9,
        typical_p=0.8,
        repetition_penalty=1.2
    )

    candidate = extract_generated_post(raw_output)

    pilot_results.append({
        "seed": seed,
        "demonstrations": demonstrations,
        "raw_output": raw_output,
        "candidate_post": candidate,
        "allowed_script": uses_allowed_script(candidate),
        "word_count": len(candidate.split())
    })

    print("=" * 80)
    print(f"Seed: {seed}")
    print("Candidate:", candidate)
    print("Allowed script:", uses_allowed_script(candidate))
    print("Word count:", len(candidate.split()))

Seed: 42
Candidate: meri behan bharatiya janani mandir wali insaan banne ke liye na, tu biwi ka gaddha bana rhe hai 🤮🤮
Allowed script: True
Word count: 19
Seed: 43
Candidate: yeh sab gandey logon ne apni behan se kaha tha "arre meri behan tumhara woh saath rehte hain jo tumhein har ghar se nikala dete hain"
Allowed script: True
Word count: 26
Seed: 44
Candidate: aisha bajwa tumhari ammi banne wale logon se zyada randi hain, aapas mein hi kuch logon ka intezaar hain 🤔.
Allowed script: True
Word count: 20
Seed: 45
Candidate: Here's a new Roman Urdu social media post that fits the "Sexist abusive language" dataset label:
Allowed script: True
Word count: 16
Seed: 46
Candidate: aaj bheji hai tumhen yeh naya video link, tum ne jo "beghairat biwi" banne wali hain, dekh sakti hain apne aap ko chup-chap puchhane ke saath ��
Allowed script: True
Word count: 27


In [60]:
CLASS_DESCRIPTIONS = {
    2: {
        "label": "religious hate speech",
        "definition": (
            "content that insults or expresses hostility towards a person "
            "or group because of religion, sect, or religious identity"
        )
    },
    3: {
        "label": "sexist content",
        "definition": (
            "content that insults, humiliates, stereotypes, or degrades "
            "a person because of gender"
        )
    },
    4: {
        "label": "profane content",
        "definition": (
            "content containing vulgar, obscene, or strongly offensive language"
        )
    }
}

In [61]:
def build_single_seed_prompt(class_id, seed_text):
    """
    Creates a literature-inspired Positive-to-Positive prompt
    using one real RUHSOLD example as the generation seed.
    """

    if class_id not in CLASS_DESCRIPTIONS:
        raise ValueError(
            f"Unsupported class_id: {class_id}. "
            f"Expected one of {list(CLASS_DESCRIPTIONS)}."
        )

    seed_text = str(seed_text).strip()

    if not seed_text:
        raise ValueError("The seed text cannot be empty.")

    class_label = CLASS_DESCRIPTIONS[class_id]["label"]
    class_definition = CLASS_DESCRIPTIONS[class_id]["definition"]

    user_prompt = f"""
You are an expert in generating synthetic data for hate speech classification.

This Roman Urdu sentence is labelled as {class_label}:

"{seed_text}"

In this dataset, {class_label} refers to {class_definition}.

Generate one new Roman Urdu sentence with the same dataset label. Preserve the informal slang, spelling variation, and social-media writing style, but express a different idea and do not copy the original sentence.

Roman Urdu means Urdu written using the English Latin alphabet. Natural English code-mixing is allowed.

Return only the new sentence.
""".strip()

    return [
        {
            "role": "user",
            "content": user_prompt
        }
    ]

In [62]:
def sample_one_seed(
    dataframe,
    class_id,
    random_state=None
):
    min_words, max_words = CLASS_WORD_LIMITS[class_id]

    class_data = dataframe[
        dataframe["label"] == class_id
    ].copy()

    class_data["word_count"] = (
        class_data["text"]
        .astype(str)
        .str.split()
        .str.len()
    )

    eligible_data = (
        class_data[
            class_data["word_count"].between(min_words, max_words)
        ]
        .drop_duplicates(subset=["text"])
    )

    if eligible_data.empty:
        raise ValueError(
            f"No eligible seed examples found for class {class_id}."
        )

    return eligible_data.sample(
        n=1,
        random_state=random_state
    )["text"].iloc[0]

In [63]:
single_seed_results = []

for seed in [42, 43, 44, 45, 46]:

    source_text = sample_one_seed(
        dataframe=train_df,
        class_id=3,
        random_state=seed
    )

    messages = build_single_seed_prompt(
        class_id=3,
        seed_text=source_text
    )

    raw_output = generate_one_sample(
        messages=messages,
        seed=seed,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        typical_p=0.8,
        repetition_penalty=1.2
    )

    candidate = extract_generated_post(raw_output)

    record = {
        "seed": seed,
        "source_text": source_text,
        "raw_output": raw_output,
        "candidate_post": candidate,
        "allowed_script": uses_allowed_script(candidate),
        "word_count": len(candidate.split())
    }

    single_seed_results.append(record)

    print("=" * 80)
    print(f"Seed: {seed}")
    print("Source:", source_text)
    print("Generated:", candidate)
    print("Allowed script:", record["allowed_script"])
    print("Word count:", record["word_count"])

Seed: 42
Source: khan sib waqae ahmaq hi kyo ki ap jesi makar awrt si shade jo kye the😢😢
Generated: "Aapne bhi tu meri behen bani thi agle saath, lekin maine usse samjh saki tau yaar 🙄🙄"
Allowed script: True
Word count: 17
Seed: 43
Source: apni maan k name pe be chona laga gya bilawal hijra
Generated: "Tu hai yeh biwi apne bete ke saath reh jati ho, par wo ladkiyan ko dhoondhta hai? Tere liye woh bhi hijra ban sakti
Allowed script: True
Word count: 24
Seed: 44
Source: ye khud sali randi hai tabhi to muh fat raha hai
Generated: "Yeh woh behen bhar ke saath rehte hain, par wo apni aukat nahin samjh sakti?"
Allowed script: True
Word count: 15
Seed: 45
Source: main sb becharay randay hain . app b free ho. nikkah kar lo.....
Generated: "Aapne bechari hui bhi nahin, tum logon ne unhe gaddar banaya hai, khud ko bharosa rakho, shaadi kar ley rehna chahi
Allowed script: True
Word count: 21
Seed: 46
Source: sasti randi.... apna modi ki english dkhi ha.. bc us ka knowledge dkha ha... 🤣 🤣 🤣 🤣 🤣
Ge

In [64]:
CLASS_DESCRIPTIONS[4] = {
    "label": "Profane language",
    "definition": (
        "A social-media post containing vulgar, obscene, or strongly "
        "offensive language. It does not need to target religion or gender."
    )
}

In [65]:
profane_results = []

for seed in [42, 43, 44, 45, 46]:

    demonstrations = sample_demonstrations(
        dataframe=train_df,
        class_id=4,
        number_of_examples=5,
        random_state=seed
    )

    messages = build_generation_prompt(
        class_id=4,
        demonstrations=demonstrations
    )

    raw_output = generate_one_sample(
        messages=messages,
        seed=seed,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        typical_p=0.8,
        repetition_penalty=1.2
    )

    candidate = extract_generated_post(raw_output)

    profane_results.append({
        "seed": seed,
        "demonstrations": demonstrations,
        "raw_output": raw_output,
        "candidate_post": candidate,
        "allowed_script": uses_allowed_script(candidate),
        "word_count": len(candidate.split())
    })

    print("=" * 80)
    print("Seed:", seed)
    print("Generated:", candidate)
    print("Allowed script:", uses_allowed_script(candidate))
    print("Word count:", len(candidate.split()))

Seed: 42
Generated: tere saathi logon ne apne beghairat banaye hain, meri khidmat ke liyae unhe zulm ka tarika dene waley ko goli maang rahe hain
Allowed script: True
Word count: 23
Seed: 43
Generated: اس میرے بھائی کو خدا نہیں مجھे لگتا ہے جو کچھا Arabic fonts
Allowed script: False
Word count: 13
Seed: 44
Generated: aaj bhi tumhara behanji logon ne diya tha jab tak raat bhar baate hue yeh meme share kiya, meri aatma ko acchi si tarah bhar g
Allowed script: True
Word count: 26
Seed: 45
Generated: aap logon ne meri performance ko bharosa di hai toh main apni aasman se nazar nikaltungi, magar ye bhenchod traffic waley logon se dhundhli me
Allowed script: True
Word count: 25
Seed: 46
Generated: aaj apne dost ne mujh ko bata diye tha k woh "bada gandu" hai, main usse poora qadar nazuk rehta hoon🤢 But now
Allowed script: True
Word count: 23


In [66]:
def build_simple_roman_urdu_prompt():
    user_prompt = """
Write one informal Roman Urdu social media post about cricket.

Roman Urdu means Urdu written using the English Latin alphabet.

The post should sound natural and conversational.

Return only the post.
""".strip()

    return [
        {
            "role": "user",
            "content": user_prompt
        }
    ]

In [67]:
diagnostic_results = []

for seed in [42, 43, 44, 45, 46]:

    messages = build_simple_roman_urdu_prompt()

    raw_output = generate_one_sample(
        messages=messages,
        seed=seed,
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        typical_p=0.8,
        repetition_penalty=1.2
    )

    candidate = extract_generated_post(raw_output)

    diagnostic_results.append({
        "seed": seed,
        "raw_output": raw_output,
        "candidate_post": candidate,
        "allowed_script": uses_allowed_script(candidate),
        "word_count": len(candidate.split())
    })

    print("=" * 80)
    print("Seed:", seed)
    print("Generated:", candidate)
    print("Allowed script:", uses_allowed_script(candidate))
    print("Word count:", len(candidate.split()))

Seed: 42
Generated: 🏏 Hey guys, how's it going? Excited for tonight's big match between Pakistan and India at the World Cup? I'm personally rooting for Sarfaraz to lead us to victory! Who are
Allowed script: True
Word count: 31
Seed: 43
Generated: 🏏 Hey guys, how's it going? Excited for tonight's big match between Pakistan and India! May the best team win, but I know my boys in green have got this in the bag ��
Allowed script: True
Word count: 34
Seed: 44
Generated: 🏏 Hey guys, who's ready for some thrilling cricket action today? Sarfaraz is leading our team with a strong determination this season! Let's show him some love in the comments below and cheer them on to
Allowed script: True
Word count: 36
Seed: 45
Generated: 🏏 Hey buddies! Excited for tonight's big match between Pakistan and India? I'm counting down the hours till we can all gather 'round our screens, munching on samosas and che
Allowed script: True
Word count: 30
Seed: 46
Generated: 🏏 Hey guys, who's ready for some thrilli

In [68]:
messages = build_simple_roman_urdu_prompt()

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(formatted_prompt)

<s> [INST] Write one informal Roman Urdu social media post about cricket.

Roman Urdu means Urdu written using the English Latin alphabet.

The post should sound natural and conversational.

Return only the post. [/INST]


In [52]:
def clean_generated_text(text):
    text = str(text).strip()

    unwanted_prefixes = [
        "New sentence:",
        "Sentence:",
        "Output:",
        "Roman Urdu:",
        "Generated sentence:"
    ]

    for prefix in unwanted_prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    text = text.strip('"').strip("'")
    text = re.sub(r"\s+", " ", text)

    return text

In [53]:
cleaned_sample = clean_generated_text(sample)

print("Raw:", sample)
print("Cleaned:", cleaned_sample)

Raw: meri behan bharatiya janani mandir wali insaan banne ke liye na, tu biwi ka gaddha bana rhe hai 🤮🤮
(My sister, you're supposed to become a Bhar
Cleaned: meri behan bharatiya janani mandir wali insaan banne ke liye na, tu biwi ka gaddha bana rhe hai 🤮🤮 (My sister, you're supposed to become a Bhar
